# Convert the CSV Data to RDF

This notebook is used to convert the CSV data into RDF format. The CSV files are read and processed, and the resulting RDF data is saved in Turtle format. The code is intentionally very verbose and not optimized for performance, as the main goal is to have a clear and understandable code that can be easily modified and extended.

## Create JSON

In the following part, all information from the different CSV files is combined into a **single deeply nested JSON** structure. This is done by iterating over the rows of the `party_csv` (which is the "main" file) and fetching all the related information from the other connected CSV files.

### Read CSV Files

In [1]:
from pathlib import Path
import pandas as pd

# path to all the CSV files relative to the location of this notebook
CSV_PATH = Path("../data/input")

# main file
party_csv = pd.read_csv(CSV_PATH / "party.csv")

# helper files
party_name_csv = pd.read_csv(CSV_PATH / "party_name.csv")
relationship_csv = pd.read_csv(CSV_PATH / "relationship.csv")
relationship_type_csv = pd.read_csv(CSV_PATH / "relationship_type.csv")
party_code_csv = pd.read_csv(CSV_PATH / "party_code.csv")
party_code_type_csv = pd.read_csv(CSV_PATH / "party_code_type.csv")

In [2]:
display(party_name_csv)

,party_name_id,party_id,notes,sources,translations_languages_code1,translations_abbreviation1,translations_name1,translations_internal_label1,translations_alternative_name1,translations_notes1,...,to_date_datetime,to_date_notes,to_date_sources,translations_languages_code4,translations_abbreviation4,translations_name4,translations_internal_label4,translations_alternative_name4,translations_notes4,translations_sources4
0,1,1,NaN,page_number_string: \r\nnotes: \r\nsource_full...,de-CH,SVP,Schweizerische Volkspartei,SVP,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,2,NaN,page_number_string: \r\nnotes: \r\nsource_full...,de-CH,SP Schweiz,Sozialdemokratische Partei der Schweiz,SP,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,3,NaN,page_number_string: 5\r\nnotes: \r\nsource_ful...,de-CH,NaN,FDP.Die Liberalen Schweiz,FDPLIB,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,4,NaN,page_number_string: \r\nnotes: \r\nsource_full...,de-CH,Mitte,Die Mitte Schweiz,Mitte,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,6,NaN,page_number_string: \r\nnotes: \r\nsource_full...,de-CH,GLP,Grünliberale Partei Schweiz,GLP,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
662,672,422,NaN,page_number_string: \r\nnotes: \r\nsource_full...,de-CH,DSP der Schweiz,Demokratisch-Soziale Partei der Schweiz,DSP CH,NaN,NaN,...,2012-01-01T12:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
663,673,423,NaN,page_number_string: \r\nnotes: \r\nsource_full...,de-CH,NaN,Demokratisch-Soziale Partei Baselland,DSP BL,NaN,NaN,...,2010-01-01T12:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
664,674,424,NaN,page_number_string: \r\nnotes: \r\nsource_full...,de-CH,DSP Graubünden,Demokratisch-Soziale Partei Graubünden,DSP GR,NaN,NaN,...,2017-01-01T12:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
665,675,425,NaN,page_number_string: \r\nnotes: \r\nsource_full...,de-CH,NaN,Bewegung Öffnung,DSP FR,NaN,NaN,...,2009-01-01T12:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Date Handling

All the dates in the CSV and after import to a pandas DataFrame are given as strings in the form of "YYYY-MM-DDT12:00:00" format (always with the time set to 12:00:00). The time part is not relevant for our use case, so we will remove it. Furthermore, the end dates are set to the date of the last valid day, so that schema:validThrough can be used:

- `schema:validFrom` means the first valid day
- `schema:validThrough` means the last valid day

Unknown dates are represented as `NaN` in the DataFrame. To check for unknown dates, we will use the `pd.isna()` function instead of checking for `None`, as the latter does not work with `NaN` values.

In [3]:
def convert_time_column(df, column_name, end_date=False):
    """
    Convert the time column to a date format and remove the time part. 
    If it is an end date, subtract one day to get the last valid day.
    Convert it back to strings in the format YYYY-MM-DD.
    """
    df[column_name] = pd.to_datetime(df[column_name], errors='coerce').dt.date
    if end_date:
        df[column_name] = df[column_name] - pd.Timedelta(days=1)
    df[column_name] = df[column_name].astype(str)
    return df

In [4]:
# convert the time columns to date format and remove the time part

party_csv = convert_time_column(party_csv, 'founding_date_datetime')
party_csv = convert_time_column(party_csv, 'dissolution_date_datetime', end_date=True)
party_name_csv = convert_time_column(party_name_csv, 'from_date_datetime')
party_name_csv = convert_time_column(party_name_csv, 'to_date_datetime', end_date=True)
party_relationship_csv = convert_time_column(relationship_csv, 'from_date_datetime')
party_relationship_csv = convert_time_column(relationship_csv, 'to_date_datetime', end_date=True)

### Create Parties JSON

In [5]:
def parse_list(s):
    """
    Parse a string representation of a list like "[1, 2, 3]" into a Python list.
    """
    if pd.isna(s) or s == "[]":
        return []
    return [int(item) for item in s.strip("[]").split(",")]

In [6]:
def is_currently_valid(day, valid_from, valid_through):
    """
    Check if a given day is within the valid range defined by valid_from and valid_through.
    valid_through can be NaN to indicate that the validity is open-ended. 
    """
    return valid_from <= str(day) and (pd.isna(valid_through) or str(day) <= valid_through)

In [ ]:
# create the main dictionary that will hold all the parties and their information
parties = {}

# for each identity (each row in the party_csv)
for _, row in party_csv.iterrows():
    
    # Create a dictionary for the current party
    party = {}
    
    # get the party_id - needed to set the key in the parties dictionary
    party_id = row["party_id"]

    # basic date information
    party["founding_date"] = row["founding_date_datetime"]
    party["dissolution_date"] = row["dissolution_date_datetime"]

    
    #### names ####
    
    # get party names
    party_name_ids = parse_list(row["party_name_id"])

    # all names are stored in a list of dictionaries
    names = []

    for party_name_id in party_name_ids:
        
        # names
        name_lang = {}
        for lang_id in range(1, 5):  # there are 4 language columns (1, 2, 3, 4)
            lang_name = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, f"translations_languages_code{lang_id}"].item()
            name = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, f"translations_name{lang_id}"].item()
            if str(lang_name) != "nan" and str(name) != "nan":
                name_lang[str(lang_name)[0:2]] = name
        
        # abbreviations
        abbr_lang = {}
        for lang_id in range(1, 5):  # there are 4 language columns (1, 2, 3, 4)
            lang_abbr = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, f"translations_languages_code{lang_id}"].item()
            abbr = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, f"translations_abbreviation{lang_id}"].item()
            if str(lang_abbr) != "nan" and str(abbr) != "nan":
                abbr_lang[str(lang_abbr)[0:2]] = abbr
        
        # dates for the name and abbreviation
        valid_from = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, "from_date_datetime"].item()
        valid_through = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, "to_date_datetime"].item()
        
        # create a name entry and add it to the names list
        name_entry = {"id": party_name_id, "name": name_lang, "abbr": abbr_lang, "valid_from": valid_from, "valid_through": valid_through}
        names.append(name_entry)

    
    # sort names by from_date
    names.sort(key=lambda x: x["valid_from"])
    
    # add names to the party dictionary
    party["names"] = names

    
    #### relationships ####

    # get relationships, they are given in two columns, relationship_id_1 and relationship_id_2.
    # They go to either shape-changing or hierarchical relationships.
    relationships = []

    shape_changing_relationship_types = ["successor-of", "split-from", "accessed"]
    hierarchical_relationship_types = ["child-of", "observer-in", "affiliated-with"]

    # get relationships from both columns and combine them into a single list
    relationship_ids = parse_list(row["relationship_id_1"]) + parse_list(row["relationship_id_2"])

    for relationship_id in relationship_ids:
        
        relationship_from = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "party_1_id"].item()
        relationship_to = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "party_2_id"].item()
        relationship_type = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "type"].item()
        relationship_label = relationship_type_csv.loc[relationship_type_csv["relationship_type_id"] == relationship_type, "identifier"].item()
        valid_from = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "from_date_datetime"].item()
        valid_through = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "to_date_datetime"].item()
        
        relationships.append(
            {"id": relationship_id, 
             "type_label": relationship_label,
             "party_from": relationship_from,
             "party_to": relationship_to,
             "valid_from": valid_from,
             "valid_through": valid_through}
        )

    # sort relationships by from_date and add them to the party dictionary
    relationships.sort(key=lambda x: x["valid_from"])
    
    # add relationships to the party dictionary
    party["relationships"] = relationships

    
    #### party identifiers ####

    # different identifiers for the same party (e.g. bfs-code, wikidata, etc.)
    party_code_ids = parse_list(row["party_code_id"])
    party_codes = []

    for party_code_id in party_code_ids:
        code = party_code_csv.loc[party_code_csv["party_code_id"] == party_code_id, "code"].item()
        code_type = party_code_csv.loc[party_code_csv["party_code_id"] == party_code_id, "party_code_type_id"].item()
        code_type_label = party_code_type_csv.loc[party_code_type_csv["party_code_type_id"] == code_type, "identifier"].item()

        code_entry = {"id": party_code_id, "code": code, "type_label": code_type_label}
        party_codes.append(code_entry)

    party_codes.append({"code": row["identifier"], "type_label": "ap-identifier"})  # add the identifier from the party.csv row

    party["party_codes"] = party_codes

    # add party to parties dictionary (finish line operations)
    parties[party_id] = party

In [38]:
import json

print(json.dumps(parties, indent=2, ensure_ascii=False))

{
  "1": {
    "founding_date": "1971-09-22",
    "dissolution_date": NaN,
    "names": [
      {
        "id": 1,
        "name": {
          "de": "Schweizerische Volkspartei",
          "fr": "Union Démocratique du Centre",
          "en": "Schweizerische Volkspartei"
        },
        "abbr": {
          "de": "SVP",
          "fr": "UDC"
        },
        "valid_from": "1971-09-22",
        "valid_through": NaN
      }
    ],
    "relationships": [
      {
        "id": 3,
        "type_label": "successor-of",
        "party_from": 1,
        "party_to": 11,
        "valid_from": "1971-09-22",
        "valid_through": NaN
      },
      {
        "id": 9,
        "type_label": "successor-of",
        "party_from": 1,
        "party_to": 18,
        "valid_from": "1971-09-22",
        "valid_through": NaN
      },
      {
        "id": 12,
        "type_label": "child-of",
        "party_from": 20,
        "party_to": 1,
        "valid_from": "1971-09-22",
        "valid_through"

## Convert the Deep JSON to RDF

### URIs

In [10]:
PARTY_IDENTITY = "https://politics.ld.admin.ch/party-id"
PARTY_VERSION = "https://politics.ld.admin.ch/party-version"
PARTY_VOCAB = "https://politics.schema.ld.admin.ch/party-vocab"
PARTY_PREFIX = "pol"

### Transformation

In [30]:
mapping_dict = {

    "affiliated-with": {
        "from": pol.affiliatedWith,
        "to": pol.affiliatedBy
    },
    "child-of": {
        "from": pol.childOf,
        "to": pol.parentOf
    },
    "observer-in": {
        "from": pol.observerIn,
        "to": pol.observedBy
    },
    "accessed": {
        "from": pol.accessed,
        "to": pol.accessedBy
    },
    "successor-of": {
        "from": pol.successorOf,
        "to": pol.predecessorOf
    },
    "split-from": {
        "from": pol.splitFrom,
        "to": pol.splitInto
    }
}

In [44]:
from rdflib import Graph, Literal, RDF, URIRef, Namespace

# create graph without binding namespaces to avoid automatic prefixing
g = Graph(bind_namespaces="none")

schema = Namespace("http://schema.org/")
g.bind("schema", schema)
xsd = Namespace("http://www.w3.org/2001/XMLSchema#")
g.bind("xsd", xsd)
dct = Namespace("http://purl.org/dc/terms/")
g.bind("dct", dct)
owl = Namespace("http://www.w3.org/2002/07/owl#")
g.bind("owl", owl)
vl = Namespace("https://version.link/")
g.bind("vl", vl)
pol = Namespace(PARTY_VOCAB)
g.bind(PARTY_PREFIX, pol)

for party_id, party in parties.items():
    
    # create identity
    identity_uri = URIRef(f"{PARTY_IDENTITY}/{party_id}")
    g.add((identity_uri, RDF.type, vl.Identity))

    # add deprecated type to identity
    if not pd.isna(party["dissolution_date"]):
        g.add((identity_uri, RDF.type, vl.Deprecated))
   
    # add Wikidata identifier if available
    wikidata_codes = [code for code in party["party_codes"] if code["type_label"] == "wikidata-code"]
    if wikidata_codes:
        wikidata_code = wikidata_codes[0]["code"]
        g.add((identity_uri, owl.sameAs, URIRef(f"https://www.wikidata.org/wiki/{wikidata_code}")))

    # add name to identity
    # the last name is the most recent (because names are ordered)
    # therefore one need to check if there are any names
    if party["names"]:

        name_identity = party["names"][-1]

        for lang, name in name_identity["name"].items():
            g.add((identity_uri, schema.name, Literal(name, lang=lang)))
        for lang, abbr in name_identity["abbr"].items():
            g.add((identity_uri, schema.alternateName, Literal(abbr, lang=lang)))

    
    # create version
    version_uri = URIRef(f"{PARTY_VERSION}/{party_id}")
    g.add((version_uri, RDF.type, vl.Version))
    g.add((version_uri, vl.identity, identity_uri))
    g.add((identity_uri, vl.version, version_uri))

    # add founding (and dissolution) date to version
    g.add((version_uri, schema.validFrom, Literal(party["founding_date"], datatype=xsd.date)))
    if not pd.isna(party["dissolution_date"]):
        g.add((version_uri, schema.validThrough, Literal(party["dissolution_date"], datatype=xsd.date)))

    # add names and abbreviations
    for name_entry in party["names"]:
        name_uri = URIRef(f"{PARTY_VERSION}/{party_id}/name/{name_entry['id']}")
        g.add((version_uri, pol.qualifiedName, name_uri))
        g.add((name_uri, RDF.type, pol.QualifiedName))

        for lang, name in name_entry["name"].items():
            g.add((name_uri, schema.name, Literal(name, lang=lang)))
        for lang, abbr in name_entry["abbr"].items():
            g.add((name_uri, schema.alternateName, Literal(abbr, lang=lang)))
        g.add((name_uri, schema.validFrom, Literal(name_entry["valid_from"], datatype=xsd.date)))
        if not pd.isna(name_entry["valid_through"]):
            g.add((name_uri, schema.validThrough, Literal(name_entry["valid_through"], datatype=xsd.date)))

    # add relationships
    for relationship in party["relationships"]:

        relationship_uri = URIRef(f"{PARTY_VERSION}/{party_id}/relationship/{relationship['id']}")
        g.add((version_uri, pol.qualifiedRelationship, relationship_uri))
        g.add((relationship_uri, RDF.type, pol.QualifiedRelationship))

        # if relationship with a time duration
        if relationship["type_label"] in ["affiliated-with", "child-of", "observer-in"]:
            g.add((relationship_uri, schema.validFrom, Literal(relationship["valid_from"], datatype=xsd.date)))

            # if it is currently valid, add the relationship to the identity as well
            if pd.isna(relationship["valid_through"]):
                if relationship["party_from"] == party_id:
                    g.add((identity_uri, mapping_dict[relationship["type_label"]]["from"], URIRef(f"{PARTY_IDENTITY}/{relationship['party_to']}")))
                else:
                    g.add((identity_uri, mapping_dict[relationship["type_label"]]["to"], URIRef(f"{PARTY_IDENTITY}/{relationship['party_from']}")))  
            else:
                g.add((relationship_uri, schema.validThrough, Literal(relationship["valid_through"], datatype=xsd.date)))

        # instantaneous relationships (shape-changing) only have a valid_from date, no valid_through date
        else:
            g.add((relationship_uri, dct.date, Literal(relationship["valid_from"], datatype=xsd.date)))

        # add the relationship to the version
        if relationship["party_from"] == party_id:
            g.add((relationship_uri, mapping_dict[relationship["type_label"]]["from"], URIRef(f"{PARTY_VERSION}/{relationship['party_to']}")))
        else:
            g.add((relationship_uri, mapping_dict[relationship["type_label"]]["to"], URIRef(f"{PARTY_VERSION}/{relationship['party_from']}")))  
            
g.serialize(destination="../data/output/parties.ttl", format="turtle")

<Graph identifier=Nb0d5ea84947a471bb90d6238a5fbdd41 (<class 'rdflib.graph.Graph'>)>